In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

import brain_probability_flux.plotting as bpp
from brain_probability_flux.dina.dynamics import ising as idyna

In [ ]:
# Results are read from data/. To plot a fresh run, set DATA_DIR to its run
# directory, for example Path("../data/2026-10-20").
DATA_DIR = bpp.DATA_DIR

In [ ]:
max_fig_width = bpp.FIG_WIDTH
max_fig_height = bpp.FIG_HEIGHT

In [ ]:
pfa, cluster_inds = bpp.aggregated_pfa(data_dir=DATA_DIR)
num_clusters = len(np.unique(cluster_inds))
color = bpp.spin_colors(num_clusters)
pfa.perform_WPCA(spin_colors=color)

In [ ]:
flux_matrices = bpp.flux_matrices_by_task(pfa)
transition_indices = bpp.get_one_bit_difference_indices(num_clusters)
# Glauber model inferred by scripts/infer_networks.py
glauber = bpp.load_results("inferred_networks", "glauber", DATA_DIR)

In [ ]:
model_flux = [
    idyna.im_to_flux(
        glauber["beta_J"][k],
        1.0,
        weight_type="glauber",
        magnetic_field=glauber["beta_h"][k],
        rate_const=float(glauber["rate_const"][k]),
    )
    for k in range(bpp.NUM_TASKS)
]
scales = []
titles = []
for k in range(bpp.NUM_TASKS):
    observed = flux_matrices[k][transition_indices]
    model = model_flux[k][transition_indices]
    # match the widest model arrow to the widest observed arrow at scale 40
    model_max = np.max(np.abs(model))
    scales.append(
        40 * np.max(np.abs(observed)) / model_max if model_max > 0 else 40
    )
    r_value = stats.pearsonr(observed, model).statistic
    titles.append(f"{pfa.task_name_list[k]}, $r={r_value:.2f}$")

fig, axs = plt.subplots(
    3, 3, figsize=(max_fig_width, max_fig_height), sharex=True, sharey=True
)
bpp.plot_pca_by_task(
    pfa,
    axs,
    scale=scales,
    color=color,
    titles=titles,
    arrows=[idyna.flux_to_flux_list(flux) for flux in model_flux],
)
bpp.save_panel(fig, "../figures/fig-s09", "fig-s09", bbox_inches=None)